# PPO Training Example on ImprovedComSatEnv

This notebook demonstrates training a PPO agent from `tensoraerospace/agent/ppo/model.py` on the `ImprovedComSatEnv` environment from `tensoraerospace/envs/comsat.py`.

**Environment features:**
- Communication satellite control via tangential thrust u2
- Tracking angular velocity theta_dot while stabilizing the orbit
- Normalized observation and action spaces [-1, 1]
- LQR-style reward function

**Algorithm features:**
- Continuous actions with Gaussian policy
- Clipped surrogate objective for stable updates
- tqdm progress bar and TensorBoard logging
- Checkpointing for training resumption

In [5]:
# Optional installs (run if needed)
# %pip install tensorboard tqdm --quiet

import os
import numpy as np
import torch

from tensoraerospace.envs import ImprovedComSatEnv
from tensoraerospace.agent.ppo.model import PPO

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("Using device:", DEVICE)

# Reproducibility
np.random.seed(42)
_ = torch.manual_seed(42)


Using device: mps


In [8]:
# Build environment
from tensoraerospace.utils import convert_tp_to_sec_tp, generate_time_period

# Time base
dt = 0.1  # Smaller timestep for satellite dynamics
_tp = generate_time_period(tn=20, dt=dt)
tps = convert_tp_to_sec_tp(_tp, dt=dt)
number_time_steps = len(_tp)

# Reference angular velocity (constant target for easier learning)
target_angular_velocity = 0.001  # rad/s
reference_signals = np.ones((1, number_time_steps)) * target_angular_velocity

# Initial state [rho (km), rho_dot (m/s), theta_dot (rad/s)]
# Start at Earth radius with target angular velocity
init_state = np.array([6371.0, 0.0, target_angular_velocity], dtype=np.float32)

# Env
env = ImprovedComSatEnv(
    initial_state=init_state,
    reference_signal=reference_signals,
    number_time_steps=number_time_steps,
    dt=dt,
    initial_thrust=0.0,
    nominal_rho=6371.0,
)

obs, info = env.reset()
print("Obs shape:", np.array(obs).shape)
print("Action space:", env.action_space)
print("Observation space:", env.observation_space)
print("\nEnvironment configuration:")
print(f"  dt: {env.dt} sec")
print(f"  Steps: {number_time_steps}")
print(f"  Duration: {number_time_steps * env.dt:.1f} sec")
print(f"  Target θ̇: {target_angular_velocity} rad/s")
print(f"  Reward scale: {env.reward_scale}")
print(f"  Survival bonus: +0.1 per step")


Obs shape: (4,)
Action space: Box(-1.0, 1.0, (1,), float32)
Observation space: Box(-1.0, 1.0, (4,), float32)

Environment configuration:
  dt: 0.1 sec
  Steps: 201
  Duration: 20.1 sec
  Target θ̇: 0.001 rad/s
  Reward scale: 0.1
  Survival bonus: +0.1 per step


In [11]:
# Create PPO agent (tuned hyperparams)
# Hyperparameters for ComSat:
# - Higher gamma (0.99) for long-term orbital stability
# - Larger rollout_len (2048) for diverse trajectory collection
# - Moderate clip_param (0.2) balances exploration and stability
# - Lower entropy_coef (0.01) reduces randomness for precise control
# - Smaller learning rates for satellite control
# - normalize_obs=True for stable learning with normalized observations

agent = PPO(
    env=env,
    gamma=0.99,                # discount factor
    max_episodes=1000,         # total training episodes
    rollout_len=2048,          # steps per rollout
    clip_pram=0.2,             # PPO clip parameter
    num_epochs=10,             # optimization epochs per rollout
    batch_size=64,             # minibatch size
    entropy_coef=0.01,         # entropy bonus coefficient
    actor_lr=3e-4,             # policy learning rate
    critic_lr=1e-3,            # value function learning rate
    gae_lambda=0.95,           # GAE lambda
    max_grad_norm=0.5,         # gradient clipping
    target_kl=0.015,           # early stopping KL threshold
    normalize_obs=True,        # normalize observations
    normalize_reward=False,    # don't normalize rewards (already scaled)
    seed=42,
)

print("PPO agent created")
# print(f"Actor network: {agent.policy_network}")
# print(f"\nValue network: {agent.value_network}")


PPO agent created


## Training

**Training recommendations:**

1. **Start simple** — use a constant target angular velocity
2. **Monitor metrics:**
   - `mean_reward` — should increase and stabilize
   - `policy_loss` — should decrease
   - `entropy` — should gradually decrease (not collapse)
3. **Early stopping** — if reward plateaus for >100 episodes
4. **Scale up** — try sinusoidal reference after convergence

In [12]:
# Train the agent
# This will take some time depending on your hardware
# Progress will be shown via tqdm progress bar
# Metrics are logged to TensorBoard in runs/ directory

print("Starting training...")
print(f"Total episodes: {agent.max_episodes}")
print(f"Rollout length: {agent.rollout_len}")
print(f"Expected total steps: ~{agent.max_episodes * agent.rollout_len // number_time_steps}")
print("\nMonitor training: tensorboard --logdir runs/\n")

# Train
agent.train()

print("\nTraining completed!")


Starting training...
Total episodes: 1000
Rollout length: 2048
Expected total steps: ~10189

Monitor training: tensorboard --logdir runs/



  1%|          | 10/1000 [00:07<12:32,  1.32it/s]


New best model! Reward: -5519.07 (episode 10)


  2%|▏         | 20/1000 [00:15<12:08,  1.35it/s]


New best model! Reward: -3730.20 (episode 20)


...


New best model! Reward: -17.13 (episode 230)


 25%|██▌       | 250/1000 [03:09<09:49,  1.27it/s]


New best model! Reward: -16.80 (episode 250)


 26%|██▌       | 255/1000 [03:13<09:24,  1.32it/s]


KeyboardInterrupt: 

In [ ]:
# Save the trained model
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

agent.save(checkpoint_dir)
print(f"Model saved to {checkpoint_dir}")

# To load later:
# agent = PPO.load(checkpoint_dir)

## Evaluating the Trained Model

Test the trained agent on a single episode and collect data for visualization.

In [ ]:
# Evaluate the trained agent
def evaluate_agent(agent, env, n_episodes=1):
    """Run agent for n_episodes and collect trajectory data"""
    all_states = []
    all_actions = []
    all_rewards = []
    all_references = []
    
    for episode in range(n_episodes):
        obs, info = env.reset()
        episode_states = []
        episode_actions = []
        episode_rewards = []
        episode_refs = []
        
        done = False
        step = 0
        
        while not done:
            # Get action from trained policy
            action = agent.act(obs, deterministic=True)
            
            # Store current state
            current_state = np.array(env.state).copy()
            episode_states.append(current_state.copy())
            episode_actions.append(action.copy())
            
            # Get reference at current step
            ref = env.reference_signal[:, env.current_step]
            episode_refs.append(ref.copy())
            
            # Step environment
            obs, reward, terminated, truncated, _ = env.step(action)
            episode_rewards.append(reward)
            
            done = terminated or truncated
            step += 1
        
        all_states.append(np.array(episode_states))
        all_actions.append(np.array(episode_actions))
        all_rewards.append(episode_rewards)
        all_references.append(np.array(episode_refs))
        
        print(f"Episode {episode + 1}:")
        print(f"  Total reward: {sum(episode_rewards):.3f}")
        print(f"  Steps completed: {step}/{number_time_steps}")
        print(f"  Mean reward: {np.mean(episode_rewards):.3f}")
        print(f"  Final state: ρ={current_state[0]:.2f} km, ρ̇={current_state[1]:.3f} m/s, θ̇={current_state[2]:.6f} rad/s")
    
    return all_states, all_actions, all_rewards, all_references

# Evaluate
print("Evaluating trained agent...\n")
states, actions, rewards, references = evaluate_agent(agent, env, n_episodes=1)

# Extract data for plotting
state_history = states[0]
action_history = actions[0]
reward_history = rewards[0]
reference_history = references[0]

print(f"\nCollected {len(state_history)} timesteps of data")


## Visualization

Plot graphs to analyze the trained agent's behavior:
- Angular velocity theta_dot and comparison with the target
- Radial position rho
- Radial velocity rho_dot
- Control signal (tangential thrust)

In [ ]:
import matplotlib.pyplot as plt

# Create time array
time = np.arange(len(state_history)) * dt

# Extract state components
rho = state_history[:, 0]           # Radial position (km)
rho_dot = state_history[:, 1]       # Radial velocity (m/s)
theta_dot = state_history[:, 2]     # Angular velocity (rad/s)
u2 = action_history[:, 0]           # Tangential thrust (N)
ref_theta_dot = reference_history[:, 0]  # Reference angular velocity

# Create subplots
fig, axes = plt.subplots(5, 1, figsize=(12, 14))
fig.suptitle('PPO Agent Performance on ComSat Environment', fontsize=16, fontweight='bold')

# 1. Angular velocity tracking
axes[0].plot(time, theta_dot * 1000, 'b-', linewidth=2, label='Actual θ̇')
axes[0].plot(time, ref_theta_dot * 1000, 'r--', linewidth=2, label='Reference θ̇')
axes[0].set_ylabel('Angular Velocity\n(mrad/s)', fontsize=11, fontweight='bold')
axes[0].legend(loc='best', fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_title('Angular Velocity Tracking', fontsize=12)

# 2. Radial position
axes[1].plot(time, rho, 'g-', linewidth=2)
axes[1].axhline(y=6371.0, color='r', linestyle='--', linewidth=1, label='Nominal (Earth radius)')
axes[1].set_ylabel('Radial Position\nρ (km)', fontsize=11, fontweight='bold')
axes[1].legend(loc='best', fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_title('Orbital Radius', fontsize=12)

# 3. Radial velocity
axes[2].plot(time, rho_dot, 'm-', linewidth=2)
axes[2].axhline(y=0, color='k', linestyle='--', linewidth=1, alpha=0.5)
axes[2].set_ylabel('Radial Velocity\nρ̇ (m/s)', fontsize=11, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].set_title('Radial Velocity', fontsize=12)

# 4. Control input
axes[3].plot(time, u2, 'orange', linewidth=2)
axes[3].axhline(y=0, color='k', linestyle='--', linewidth=1, alpha=0.5)
axes[3].set_ylabel('Tangential Thrust\nu₂ (N)', fontsize=11, fontweight='bold')
axes[3].grid(True, alpha=0.3)
axes[3].set_title('Control Input', fontsize=12)

# 5. Reward
axes[4].plot(time, reward_history, 'c-', linewidth=2)
axes[4].set_xlabel('Time (s)', fontsize=11, fontweight='bold')
axes[4].set_ylabel('Reward', fontsize=11, fontweight='bold')
axes[4].grid(True, alpha=0.3)
axes[4].set_title('Instantaneous Reward', fontsize=12)

plt.tight_layout()
plt.show()

# Print statistics
print("\n" + "="*60)
print("PERFORMANCE STATISTICS")
print("="*60)
print(f"\nAngular Velocity (θ̇):")
print(f"  Mean error: {np.mean(np.abs(theta_dot - ref_theta_dot)) * 1000:.4f} mrad/s")
print(f"  Max error:  {np.max(np.abs(theta_dot - ref_theta_dot)) * 1000:.4f} mrad/s")
print(f"  RMS error:  {np.sqrt(np.mean((theta_dot - ref_theta_dot)**2)) * 1000:.4f} mrad/s")

print(f"\nRadial Position (ρ):")
print(f"  Mean:  {np.mean(rho):.2f} km")
print(f"  Std:   {np.std(rho):.2f} km")
print(f"  Range: [{np.min(rho):.2f}, {np.max(rho):.2f}] km")
print(f"  Deviation from nominal: {np.mean(np.abs(rho - 6371.0)):.2f} km")

print(f"\nRadial Velocity (ρ̇):")
print(f"  Mean: {np.mean(rho_dot):.3f} m/s")
print(f"  Std:  {np.std(rho_dot):.3f} m/s")
print(f"  Max:  {np.max(np.abs(rho_dot)):.3f} m/s")

print(f"\nControl Input (u₂):")
print(f"  Mean: {np.mean(u2):.3f} N")
print(f"  Std:  {np.std(u2):.3f} N")
print(f"  Max:  {np.max(np.abs(u2)):.3f} N")

print(f"\nReward:")
print(f"  Total:  {np.sum(reward_history):.3f}")
print(f"  Mean:   {np.mean(reward_history):.3f}")
print(f"  Min:    {np.min(reward_history):.3f}")
print(f"  Max:    {np.max(reward_history):.3f}")
print("="*60)


## Additional Experiments

After successful training on a constant target, try:

### 1. Switch to a sinusoidal target
```python
from tensoraerospace.signals.standard import sinusoid_vertical_shift
ref = sinusoid_vertical_shift(tp, frequency=0.1  # was 0.01 index-based; real Hz = 0.01/dt = 0.1, amplitude=0.5, vertical_shift=1.0)
```

### 2. Adjust hyperparameters
- Increase `n_epochs` for better sample efficiency
- Adjust `clip_range` (0.1-0.3) for exploration-exploitation balance
- Try different `learning_rate` schedules

### 3. Longer training
- Increase `num_episodes` to 2000+ for more complex trajectories